In [6]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
pip install groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load API key from .env file
load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_recommendation(row):
    prompt = f"""You are a hospital supply chain analyst. Based on the following inventory data, write a 2-3 sentence actionable recommendation.

Product: {row['Product_Name']}
Branch: {row['Branch']}
Current Stock: {row['Current_Stock']} units
Average Weekly Usage: {row['Avg_Weekly_Usage']} units
Weeks of Supply Remaining: {row['Weeks_of_Supply']} weeks
Stock Status: {row['Stock_Status']}
Variance from Reorder Point: {row['Variance_Pct']}%
Reorder Point: {row['Reorder_Point']} units

Write a specific, actionable recommendation. Include urgency level and suggested reorder quantity."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )
    return response.choices[0].message.content

In [20]:
import time

recommendations = []

for i, (_, row) in enumerate(df.iterrows()):
    print(f"Processing {i+1}/47: {row['Product_Name']} | {row['Branch']}")
    rec = generate_recommendation(row)
    recommendations.append(rec)
    time.sleep(0.5)  # Small delay to avoid rate limits

df['AI_Recommendation'] = recommendations

# Save to CSV
df.to_csv('../reports/inventory_recommendations.csv', index=False)

# Also save to MySQL
df.to_sql('branch_inventory', con=engine, if_exists='replace', index=False)

print(f"\n Recommendations generated for all {len(df)} flagged items")
print(f"Saved to reports/inventory_recommendations.csv")

Processing 1/47: Bandage Rolls 4 inch | West Branch
Processing 2/47: Syringes 5ml with Needle | West Branch
Processing 3/47: Urine Collection Cups | North Branch
Processing 4/47: Infusion Sets with Filter | North Branch
Processing 5/47: N95 Respirators (Box/20) | North Branch
Processing 6/47: Oxygen Masks Adult | North Branch
Processing 7/47: Isolation Gowns Disposable | South Branch
Processing 8/47: Pulse Oximeter Probes Disposable | South Branch
Processing 9/47: Gauze Swabs Sterile 4x4 (Pack/10) | South Branch
Processing 10/47: Surgical Face Masks (Box/50) | East Branch
Processing 11/47: IV Catheters 18G | East Branch
Processing 12/47: Tongue Depressors (Pack/100) | East Branch
Processing 13/47: N95 Respirators (Box/20) | West Branch
Processing 14/47: IV Tubing Sets | West Branch
Processing 15/47: Oxygen Masks Adult | West Branch
Processing 16/47: Disposable Bed Sheets | West Branch
Processing 17/47: Antiseptic Wipes (Pack/100) | West Branch
Processing 18/47: Syringes 5ml with Needle